# Chapter 17 — AI-SPM — LLM 운영 감사·이상 탐지

> **Chapter 17 · Part III**

모델 인벤토리를 관리하고, 비용 이상치를 z-score 로, 형식·출처 위반율(citation/schema violation rate)을 임계값으로 각각 탐지합니다.

## 실습 목표
1. LLM 운영 인벤토리를 정의하고, 비용 시계열에서 z-score 이상치를 탐지합니다.
2. 같은 질문을 두 프롬프트 정책(형식·출처 강제 v2 / 제약 없는 구 정책 v1)으로 실제 Claude API 에 물어, 응답의 **형식·출처 위반율(citation/schema)** 을 실측합니다.
3. 구 정책 응답이 운영 스트림에 섞여 드는 비율을 0·5·10·25% 로 바꿔 가며, AI-SPM 의 5% 경보가 언제 켜지는지 확인합니다.
4. 빈 응답·`max_tokens` 중단은 환각이 아니라 **운영 오류**로 따로 분류합니다.

> **주의:** 비용 이상 탐지 데모는 표준 라이브러리만으로 동작합니다. 위반율 실습은 `ANTHROPIC_API_KEY` 가 있으면 실제 Claude API 호출로, 없으면 mock 응답으로 진행합니다.

In [ ]:
# 표준 라이브러리 + anthropic SDK 로 동작 — 설치 불필요(Colab 사전 설치)
# !pip install -q anthropic python-dotenv

In [ ]:
INVENTORY = [
  # daily_cost: 최근값이 과거 분포에서 튀면 이상 (claude-log-triage 는 마지막 날 급증)
  {'model': 'claude-log-triage', 'prompts': 3, 'daily_cost': [10, 11, 9, 42]},
  {'model': 'claude-rag-analyst', 'prompts': 5, 'daily_cost': [20, 19, 21, 20]},
  # 경계 사례 — 하루씩 서서히 오른다. 단일 급증이 아니라 완만한 상승은 z-score 가 놓치기 쉽다.
  {'model': 'claude-soar-agent', 'prompts': 4, 'daily_cost': [10, 12, 14, 16, 18, 20]},
]

---
## Step 1. z-score 이상치 탐지

최근 값이 과거 분포에서 크게 벗어나면 이상으로 표시합니다. 완만하게 상승하는 경계 사례는 단일 급증보다 z-score 가 낮게 나오는 것도 함께 확인합니다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: def detect_anomaly(series, z=2.5):
for m in INVENTORY:
    flag, sc = detect_anomaly(m['daily_cost'])
    print(f"{m['model']:20} 비용 z={sc:6} {'[!] 이상' if flag else 'OK'}")
print("\n경계 사례(claude-soar-agent): 완만한 상승은 z-score 가 임계(2.5) 아래라 놓친다 —")
print("이런 추세는 z-score 대신 7일 baseline 대비 누적 증가율로 따로 감시해야 한다.")

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
def detect_anomaly(series, z=2.5):
    hist, last = ...
    mean = ...
    var = ...
    std = ...
    score = ...
    return ...
```
</details>


---
## Step 2. 형식·출처 위반율(citation/schema) 실측

§17.3.4 의 `no_citation_rate` / `schema_violation_rate` 를 하나의 위반 판정으로 합쳐 실측합니다.
같은 보안 질문 8개를 두 프롬프트 정책으로 Claude API 에 실제로 묻습니다.
- `v2_strict` — `{"answer": ..., "citation": ...}` JSON 스키마와 공인 출처(OWASP·NIST·MITRE 등) 표기를 강제
- `v1_legacy` — 형식 제약 없는 자유 응답(스키마 변경 이전의 구 프롬프트를 시뮬레이션)

> **운영 오류 분리:** Opus 5 는 기본적으로 추론이 켜져 있어, `max_tokens` 를 너무 낮게 주면 추론 토큰이 먼저 소진돼 **빈 응답**(`stop_reason='max_tokens'`)이 온다. 안전 분류기가 거절하면(`stop_reason='refusal'`) 역시 빈 응답이 온다. 둘 다 "환각"이 아니라 **운영 오류**이므로 위반율 계산에서 빼고 따로 집계한다. 아래 호출은 JSON 산출이 목적이라 추론을 끄고(`thinking={'type':'disabled'}`) `max_tokens` 를 넉넉히(1024) 준다.

In [ ]:
import os, json, re
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.environ.get('ANTHROPIC_API_KEY')
USE_REAL_API = bool(API_KEY)

QUESTIONS = [
    'OWASP Top 10 의 인젝션(Injection) 항목은 무엇을 의미하는가?',
    'MITRE ATT&CK 기법 T1055(Process Injection) 는 무엇인가?',
    'CVE-2021-44228(Log4Shell) 은 어떤 취약점인가?',
    'NIST SP 800-53 AC-2 통제 항목은 무엇을 요구하는가?',
    'RFC 6749(OAuth 2.0) 의 인가 코드(Authorization Code) 흐름을 설명하라.',
    'CWE-79(XSS) 는 어떤 유형의 취약점인가?',
    'ISO/IEC 27001 부속서 A.8 은 무엇을 다루는가?',
    'SANS Top 25 중 하나인 CWE-89(SQL Injection) 를 설명하라.',
]

SYSTEM_V2_STRICT = (
    '아래 질문에 답하라. 반드시 다음 JSON 형식으로만 응답하라 (다른 텍스트·코드펜스 금지):\n'
    '{"answer": "<1~2문장 답변>", "citation": "<근거 표준/문서명, 예: OWASP Top 10, NIST SP 800-53, CVE-2023-1234>"}'
)
SYSTEM_V1_LEGACY = '아래 질문에 1~2문장으로 자유롭게 답하라. 형식 제약 없음, 출처 표기 불필요.'

def call_claude(question, system_prompt):
    # JSON 산출이 목적 — 추론을 끄고(비용·지연 절감) max_tokens 를 넉넉히 준다.
    # max_tokens 를 낮게 주면 추론이 켜졌을 때 빈 응답이 온다(운영 오류).
    resp = client.messages.create(
        model='claude-opus-5',
        max_tokens=1024,
        thinking={'type': 'disabled'},
        output_config={'effort': 'low'},
        system=system_prompt,
        messages=[{'role': 'user', 'content': question}],
    )
    text = ''.join(b.text for b in resp.content if b.type == 'text')
    return text, resp.stop_reason

def mock_call(question, policy_version):
    """API 키 없을 때 대체 — v2 는 스키마 준수, v1 은 자유 산문(스키마 위반)."""
    if policy_version == 'v2_strict':
        return json.dumps({'answer': f'{question[:20]}... 관련 표준 정의.', 'citation': 'NIST SP 800-53'}), 'end_turn'
    return f'{question[:20]}... 에 대한 간단한 설명입니다.', 'end_turn'

def run_job(job):
    i, policy_version, q, system_prompt = job
    op_error = None
    if USE_REAL_API:
        try:
            text, stop = call_claude(q, system_prompt)
            if not text.strip():
                op_error = f'빈 응답 (stop_reason={stop})'
        except Exception as e:
            text, op_error = '', f'API 오류: {type(e).__name__}'
    else:
        text, _ = mock_call(q, policy_version)
    return {'id': i, 'policy_version': policy_version, 'question': q,
            'raw_text': text, 'op_error': op_error}

jobs = []
for i, q in enumerate(QUESTIONS):
    jobs.append((i, 'v2_strict', q, SYSTEM_V2_STRICT))
    jobs.append((i, 'v1_legacy', q, SYSTEM_V1_LEGACY))

if USE_REAL_API:
    import anthropic
    client = anthropic.Anthropic(api_key=API_KEY)
    with ThreadPoolExecutor(max_workers=6) as ex:
        RESPONSES = list(ex.map(run_job, jobs))
    print(f'Claude API 로 실제 응답 {len(RESPONSES)}건 생성 (Real API mode)')
else:
    RESPONSES = [run_job(job) for job in jobs]
    print(f'Mock 모드로 응답 {len(RESPONSES)}건 생성 (API key 없음)')

op_errors = [r for r in RESPONSES if r['op_error']]
print(f'운영 오류(빈 응답·API 오류): {len(op_errors)}건 — 위반율 계산에서 제외')
for r in op_errors:
    print(f"  - [{r['policy_version']}] {r['question'][:24]}... -> {r['op_error']}")

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: CITATION_PATTERN = ...

# 운영 오류를 제외한 응답만 위반 판정 대상 — "정직한 분모"
valid = [r for r in RESPONSES if not r['op_error']]
print('정책 버전별 형식·출처 위반율:')
for pv in ('v2_strict', 'v1_legacy'):
    sub = [r for r in valid if r['policy_version'] == pv]
    v = sum(is_violation(r) for r in sub)
    print(f'  {pv:10} 위반 {v}/{len(sub)}')

# 좋은 정책(v2) 은 거의 0, 구 정책(v1) 은 거의 전부 위반 — 이 자체는 설계상 당연하다.
# 그래서 "v2 대 v1 위반율" 이 아니라, 아래처럼 구 정책 응답의 혼입 비율을 지표로 삼는다.

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
CITATION_PATTERN = ...

def is_violation(resp: ...
    ...
    text = ...
    fence = ...
    if fence:
        text = ...
    try:
        obj = ...
    except (json.JSONDecodeError, ValueError):
        return ...
    if not isinstance(obj, dict) or 'answer' not in obj or 'citation' not in obj:
        return ...
    if not CITATION_PATTERN.search(str(obj.get('citation', ''))):
        return ...
    return ...
```
</details>


---
## Step 3. 구 정책 혼입 비율 → 5% 경보 발동 지점

`v2 대 v1 의 위반율` 자체는 설계로 정해집니다 — v1 은 출처·형식을 요구하지 않으니 위반이 당연합니다.
운영에서 정작 궁금한 것은 **프롬프트 정책을 v2 로 바꾼 뒤에도 구 정책(v1) 응답이 스트림에 얼마나 섞여 드는가**,
그리고 AI-SPM 의 5% 임계가 그 오염을 언제 잡아내는가입니다.

실제로 받은 v2·v1 응답을 풀로 삼아, 구 정책 혼입 비율 `p` 를 0·5·10·25% 로 바꿔 가며 스트림을 구성하고 위반율을 잽니다.

In [ ]:
def alert(msg):
    print(f'  [ALERT] {msg}')

THRESHOLD = 0.05  # AI-SPM 형식·출처 위반율 경보 임계 (§17.5)

compliant_pool = [r for r in valid if r['policy_version'] == 'v2_strict' and not is_violation(r)]
legacy_pool    = [r for r in valid if r['policy_version'] == 'v1_legacy']  # 구 정책 = 위반원
print(f'준법 응답 풀(v2, 위반 아님): {len(compliant_pool)}건 / 구 정책 풀(v1): {len(legacy_pool)}건\n')

def build_stream(p, n=200):
    """구 정책(v1) 응답을 비율 p 로 섞은 n건 스트림 — 실제 받은 응답을 순환 사용."""
    n_legacy = round(p * n)
    stream = []
    for i in range(n):
        pool = legacy_pool if i < n_legacy else compliant_pool
        stream.append(pool[i % len(pool)])
    return stream

print(f'{"구 정책 혼입 p":>14} | {"위반율":>7} | 5% 경보')
print('-' * 40)
for p in (0.0, 0.05, 0.10, 0.25):
    stream = build_stream(p)
    rate = sum(is_violation(r) for r in stream) / len(stream)
    verdict = '켜짐 [!]' if rate > THRESHOLD else '꺼짐'
    print(f'{p:>13.0%} | {rate:>7.1%} | {verdict}')
    if rate > THRESHOLD:
        alert(f'형식·출처 위반율 임계 초과 — {rate:.1%} > {THRESHOLD:.0%} (구 정책 응답 혼입 의심)')

print('\n해석: 혼입 0~5% 는 경보가 꺼져 있고, 10% 를 넘으면 켜진다.')
print('임계(5%) 는 §17.5 의 출발값이며, 7일 baseline 에 맞춰 조정하는 설계 파라미터다.')

---
## 정리
- AI-SPM 은 LLM 운영 자체를 감사하는 메타 계층입니다 — 비용·형식·출처 위반·프롬프트 변경을 감시합니다.
- 비용 anomaly 는 z-score 로 단일 급증을 잡지만, 완만한 상승은 baseline 대비 추세로 따로 봐야 합니다.
- 형식·출처 위반율(citation/schema)은 정책 자체의 위반율이 아니라 **구 정책 응답의 혼입 비율**을 감시 지표로 삼아야 의미가 있습니다 — 5% 임계는 혼입 10% 부근에서 경보를 켰습니다.
- 빈 응답·`max_tokens` 중단·`refusal` 은 환각이 아니라 운영 오류로 분리 집계했습니다 — `max_tokens` 를 넉넉히 주고 text 블록만 읽으면 애초에 발생하지 않습니다.
- 실제 감사 파이프라인은 labs 와 ch27 을 참고하세요.

> **다음 챕터:** Continuous Adversarial Testing.